# Three fourth-order models over 200 cycles: visualization

This notebook is the presentation-only half of the comparison. It loads `results.csv`, reconstructs the aligned trajectories and diagnostics, and produces every table, figure, conclusion, and animation without running an integrator.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

from diagnostics import load_five_method_comparison_csv
from diagnostics.paths import find_project_root
from dynamics import GuidingCenterDynamics
from potential import load_gc2d_h5_potential
implicit_method_names = ("BM4Implicit", "GaussLegendre4")
from visualization import (
    animate_implicit_method_trajectories,
    display_animation,
    display_records_table,
    plot_accuracy_runtime_tradeoff,
    plot_accuracy_summary,
    plot_energy_accuracy_over_time,
    plot_implicit_method_iterations,
    plot_runtime_comparison,
    plot_trajectory_accuracy_over_time,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## Load the persisted calculation

The CSV is the boundary between AWS computation and local analysis. Its first row identifies the schema and carries the complete numerical configuration; its remaining columns retain every saved trajectory and plotted diagnostic.

In [ ]:
project_root = find_project_root(Path.cwd())
notebook_directory = (
    project_root
    / "notebooks/developements/energy/compare_three_order4_models_bm4_gauss_legendre4_rk4_200_steps_per_cycle"
)
results_path = notebook_directory / "results.csv"
result = load_five_method_comparison_csv(results_path)

experiment_metadata = result.metadata["experiment"]
potential_specification = experiment_metadata["potential"]
data_path = project_root / potential_specification["source_path"]
potential = load_gc2d_h5_potential(
    data_path,
    B=float(potential_specification["magnetic_field"]),
    characteristic_length=float(potential_specification["characteristic_length"]),
    indx=tuple(potential_specification["mode_selection"]),
    interpolation_order=int(potential_specification["interpolation_order"]),
)
effective_potential = GuidingCenterDynamics(
    potential,
    rho=float(result.metadata["config"]["rho"]),
).effective_potential

summaries = result.summaries()
nonlinear_summaries = result.nonlinear_work_summaries()
trajectory_count = int(result.metadata["particle_count"])
initial_state = next(iter(result.solutions.values())).source.initial_state
assert initial_state is not None
assert result.reference.times.size == int(result.metadata["sample_count"])
assert all(
    np.array_equal(solution.t, result.reference.times)
    for solution in result.solutions.values()
)

display(Markdown(
    f"Loaded **{results_path.relative_to(project_root)}** "
    f"({results_path.stat().st_size / 1024**2:.2f} MiB): "
    f"**{trajectory_count} trajectories**, **{result.reference.times.size} saved states**, "
    f"and **{len(result.solutions)} methods**. No integration was executed."
))
assert tuple(result.solutions) == ("BM4Implicit", "GaussLegendre4", "RK4")
assert result.metadata["config"]["integration_step"] == 0.005
step_count = int(next(iter(result.solutions.values())).diagnostics["step_count"])
step_times = np.linspace(result.reference.times[0], result.reference.times[-1], step_count + 1)[1:]


## Initial-condition coverage

Trajectory 1 is placed near, but not exactly at, the geometric center of the periodic cell and is highlighted with a gold star. Its offset is $(0.08,-0.06)$ times the cell width, giving a radial displacement of $0.10$ cell widths. Trajectories 2 and 3 retain their seeded Latin-hypercube positions.

In [ ]:
x0 = initial_state[:trajectory_count]
y0 = initial_state[trajectory_count:]
grid = potential.grid
field_at_zero = np.asarray(potential.evaluate_grid(0.0), dtype=float)

figure, axis = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
image = axis.imshow(
    field_at_zero.T,
    origin="lower",
    extent=(grid.xmin, grid.xmin + grid.period, grid.ymin, grid.ymin + grid.period),
    cmap="RdBu_r",
    aspect="equal",
)
axis.scatter(
    x0[1:],
    y0[1:],
    s=90,
    facecolors="none",
    edgecolors="black",
    linewidths=1.5,
    label="Distributed trajectories",
)
axis.scatter(
    x0[0],
    y0[0],
    s=180,
    marker="*",
    facecolor="gold",
    edgecolor="black",
    linewidth=1.0,
    label="Trajectory 1: near center",
)
for index, (x_value, y_value) in enumerate(zip(x0, y0, strict=True), start=1):
    axis.annotate(str(index), (x_value, y_value), xytext=(5, 5), textcoords="offset points")
axis.set(title="One near-center and two distributed initial conditions", xlabel="$x$", ylabel="$y$")
axis.legend(loc="upper right")
figure.colorbar(image, ax=axis, label="$\Phi(0,x,y)$")
plt.show()

initial_rows = tuple(
    SimpleNamespace(trajectory=index, x=x_value, y=y_value)
    for index, (x_value, y_value) in enumerate(zip(x0, y0, strict=True), start=1)
)
display_records_table(
    initial_rows,
    columns=(("trajectory", "Trajectory", "d"), ("x", "Initial x", ".8f"), ("y", "Initial y", ".8f")),
)

## Persisted integration record

The following table reports the references and alternating timing campaign recorded by the calculation notebook. Reading this section performs no numerical integration.

In [ ]:
display(Markdown(
    f"The persisted study runtime was **{result.total_study_runtime_seconds:.3f} s** "
    f"for timed runs using saved references. The DOP853/Radau space-time RMS "
    f"reference discrepancy is **{result.reference.time_integrated_rms_floor:.3e}**."
))

display(Markdown("### Completed execution log"))
display_records_table(
    result.execution_log,
    columns=(
        ("phase", "Phase", None),
        ("repeat", "Repeat", "d"),
        ("method_label", "Method", None),
        ("trajectory_count", "Trajectories", "d"),
        ("step_count", "Effective steps", "d"),
        ("runtime_seconds", "Runtime [s]", ".4f"),
    ),
)

## Precision, runtime, nonlinear work and energy summary

Minimum-image periodic trajectory errors and physical-Hamiltonian errors use the original 10,001 reference times. Runtime statistics describe elapsed time under concurrent execution and therefore include contention between the three selected models. Newton statistics include every one of the 40,000 complete steps for BM4 and Gauss–Legendre4. RK4 is explicit.


In [ ]:
display_records_table(
    summaries,
    columns=(
        ("method_label", "Method", None),
        ("solver", "Solver", None),
        ("trajectory_count", "Trajectories", "d"),
        ("time_integrated_rms_distance", "Space-time RMS error", ".4e"),
        ("final_rms_distance", "Final RMS error", ".4e"),
        ("runtime_seconds", "Median runtime [s]", ".4f"),
        ("runtime_first_quartile_seconds", "Runtime Q1 [s]", ".4f"),
        ("runtime_third_quartile_seconds", "Runtime Q3 [s]", ".4f"),
        ("time_integrated_rms_energy_error", "Space-time RMS energy error", ".4e"),
        ("maximum_absolute_energy_error", "Maximum absolute energy error", ".4e"),
    ),
)

display(Markdown("### Newton work for implicit methods"))
display_records_table(
    nonlinear_summaries,
    columns=(
        ("method_label", "Method", None),
        ("nonlinear_solves_per_step", "Nonlinear solves/step", "d"),
        ("mean_newton_iterations", "Mean Newton corrections/step", ".3f"),
        ("maximum_newton_iterations", "Max Newton corrections/step", "d"),
        ("total_newton_iterations", "Total Newton corrections", "d"),
        ("mean_residual_evaluations", "Mean residual evaluations/step", ".3f"),
        ("total_residual_evaluations", "Total residual evaluations", "d"),
        ("maximum_residual_to_tolerance", "Max residual/tolerance", ".3e"),
    ),
)

## Long-time trajectory precision and cost

The first figure shows the full error history through 200 cycles. Once trajectories separate by an appreciable fraction of the periodic domain, pointwise distance becomes a loss-of-phase indicator rather than a local truncation-error measurement. The summary and cost plots complement that history with integrated and endpoint values. The final two-panel figure compares absolute median runtime with its interquartile range and slowdown relative to the fastest method.

In [ ]:
plot_trajectory_accuracy_over_time(
    result.reference.times,
    result.accuracy,
    reference_floor=result.reference.time_integrated_rms_floor,
)
plt.show()

plot_accuracy_summary(summaries)
plt.show()

plot_accuracy_runtime_tradeoff(summaries)
plt.show()

plot_runtime_comparison(summaries)
plt.show()

## Newton work on every effective step

Diagnostics cover all 40,000 complete steps, separately from the saved trajectory grid. A residual-to-tolerance ratio at or below one certifies nonlinear acceptance.


In [ ]:
implicit_solutions = {
    name: result.solutions[name]
    for name in implicit_method_names
}
plot_implicit_method_iterations(implicit_solutions)
plt.show()

## Projection-multiplier norm for BM4

BM4 solves for one reduced projection multiplier at every complete step. The table and history report its infinity norm over all 40,000 steps. Gauss–Legendre4 and RK4 do not use this multiplier.


In [ ]:
projection_method_names = (
    "BM4Implicit",
)
projection_multiplier_norms = {
    name: np.asarray(
        result.solutions[name].diagnostics["projection_multiplier_norms"],
        dtype=float,
    )
    for name in projection_method_names
}
expected_multiplier_shape = (step_count,)
for name, values in projection_multiplier_norms.items():
    assert values.shape == expected_multiplier_shape
    assert np.all(np.isfinite(values))
    assert np.all(values >= 0.0)

multiplier_summary_rows = tuple(
    SimpleNamespace(
        method_name=name,
        method_label=next(
            row.method_label for row in summaries if row.method_name == name
        ),
        mean_multiplier_norm=float(np.mean(values)),
        rms_multiplier_norm=float(np.sqrt(np.mean(values**2))),
        maximum_multiplier_norm=float(np.max(values)),
        final_multiplier_norm=float(values[-1]),
    )
    for name, values in projection_multiplier_norms.items()
)
display_records_table(
    multiplier_summary_rows,
    columns=(
        ("method_label", "Method", None),
        ("mean_multiplier_norm", r"Mean $\|\mu_n\|_\infty$", ".4e"),
        ("rms_multiplier_norm", r"RMS $\|\mu_n\|_\infty$", ".4e"),
        ("maximum_multiplier_norm", r"Max $\|\mu_n\|_\infty$", ".4e"),
        ("final_multiplier_norm", r"Final $\|\mu_n\|_\infty$", ".4e"),
    ),
)

figure, axis = plt.subplots(figsize=(12, 4.8), constrained_layout=True)
for name, values in projection_multiplier_norms.items():
    method_label = next(
        row.method_label for row in multiplier_summary_rows if row.method_name == name
    )
    axis.semilogy(
        step_times,
        np.maximum(values, np.finfo(float).tiny),
        label=method_label,
    )
axis.set(
    title="Projection-multiplier norm over 200 cycles",
    xlabel=r"$t_{n+1}$",
    ylabel=r"$\|\mu_n\|_\infty$",
)
axis.legend()
plt.show()

## Physical-energy evolution

The measured potential contains a time-dependent mode, so physical energy is not conserved. The relevant diagnostic is reproduction of the DOP853 energy history. The top panel shows instantaneous particle-RMS Hamiltonian error; the bottom panel shows the running worst absolute error over all saved times and particles.

In [ ]:
plot_energy_accuracy_over_time(
    result.reference.times,
    result.energy_accuracy,
    reference_energy_errors=result.reference_energy_errors,
)
plt.show()

## Downsampled trajectory animation

The numerical results retain all 10,001 saved states. To keep the embedded animation compact, 201 uniformly distributed frames summarize the 200-cycle evolution at ten frames per second: approximately one frame per cycle.

In [ ]:
animation_frames = min(201, result.reference.times.size)
animation_fps = 10
trajectory_animation = animate_implicit_method_trajectories(
    effective_potential,
    result.solutions,
    frames=animation_frames,
    interval=int(round(1000.0 / animation_fps)),
    repeat=True,
    title_family="numerical methods",
)
display_animation(trajectory_animation, embed_limit_mb=100.0)

## Data-driven conclusion

The following statements are generated from the measured run rather than hard-coded.

In [ ]:
fastest = min(summaries, key=lambda row: row.runtime_seconds)
most_accurate = min(summaries, key=lambda row: row.time_integrated_rms_distance)
best_energy = min(summaries, key=lambda row: row.time_integrated_rms_energy_error)
minimum_total_newton_iterations = min(
    row.total_newton_iterations for row in nonlinear_summaries
)
smallest_peak_multiplier = min(
    multiplier_summary_rows, key=lambda row: row.maximum_multiplier_norm
)
least_newton_work = tuple(
    row
    for row in nonlinear_summaries
    if row.total_newton_iterations == minimum_total_newton_iterations
)
least_newton_work_labels = " and ".join(row.method_label for row in least_newton_work)
newton_iteration_suffix = " each" if len(least_newton_work) > 1 else ""
rk4_summary = next(row for row in summaries if row.method_name == "RK4")
display(Markdown(
    "\n".join((
        f"- **Fastest median integration:** {fastest.method_label} ({fastest.runtime_seconds:.4f} s).",
        f"- **Smallest space-time RMS trajectory error:** {most_accurate.method_label} ({most_accurate.time_integrated_rms_distance:.4e}).",
        f"- **Smallest space-time RMS physical-energy error:** {best_energy.method_label} ({best_energy.time_integrated_rms_energy_error:.4e}).",
        f"- **Fewest total Newton corrections:** {least_newton_work_labels} ({minimum_total_newton_iterations:d}{newton_iteration_suffix}).",
        f"- **Smallest peak projection-multiplier norm:** {smallest_peak_multiplier.method_label} ({smallest_peak_multiplier.maximum_multiplier_norm:.4e}).",
        f"- **Classical RK4 long-time result:** trajectory RMS {rk4_summary.time_integrated_rms_distance:.4e}, energy RMS {rk4_summary.time_integrated_rms_energy_error:.4e}, median runtime {rk4_summary.runtime_seconds:.4f} s.",
    ))
))
